# Three optional reviewer analyses

**A. Decision curve analysis.** Net benefit across threshold probabilities, against treat-all and treat-none. Computed on *recalibrated* predictions, because net benefit depends on calibration and the raw weighted scores are displaced by about 2.0 on the log-odds scale.

**B. First-encounter-only sensitivity.** Restricting to each patient's first encounter removes repeated measures entirely, which directly addresses the concern that a readmission encounter can itself serve as a later index encounter.

**C. SMOTE neighbour sensitivity.** Whether the magnitude of the pre-split inflation depends on k.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Then Run all. About 15 minutes.

In [ ]:
#@title 1. Environment and cohort (keeps encounter_id for ordering)
!pip install -q xgboost==3.2.0 scikit-learn==1.8.0 imbalanced-learn==0.14.1 ucimlrepo 2>/dev/null
import numpy as np, pandas as pd, sklearn, xgboost as xgb, warnings, ssl, urllib.request
warnings.filterwarnings('ignore')
try:
    d0=xgb.DMatrix(np.zeros((16,3)),label=np.array([0,1]*8))
    xgb.train({'tree_method':'hist','device':'cuda'},d0,num_boost_round=1); USE_GPU=True
except Exception: USE_GPU=False
print('sklearn',sklearn.__version__,'| xgboost',xgb.__version__,'| GPU',USE_GPU)

ctx=ssl.create_default_context(); ctx.check_hostname=False; ctx.verify_mode=ssl.CERT_NONE
_o=urllib.request.urlopen
urllib.request.urlopen=lambda *a,**k:_o(*a,context=ctx,**{kk:vv for kk,vv in k.items() if kk!='context'})
from ucimlrepo import fetch_ucirepo
d=fetch_ucirepo(id=296)
raw=pd.concat([p for p in [d.data.ids,d.data.features,d.data.targets] if p is not None],axis=1)

EXPIRED={11,13,14,19,20,21}
DR={'has_diabetes_dx':[(250,250.99)],'has_circulatory_dx':[(390,459)],'has_respiratory_dx':[(460,519)],
    'has_renal_dx':[(580,629)],'has_digestive_dx':[(520,579)],'has_infectious_dx':[(1,139)],
    'has_injury_dx':[(800,999)],'has_neoplasm_dx':[(140,239)],'has_symptoms_dx':[(780,799)]}
u=raw.replace('?',np.nan).copy(); u=u[~u['discharge_disposition_id'].isin(EXPIRED)].copy()
for c in ['diag_1','diag_2','diag_3']:
    u[c]=pd.to_numeric(u[c].astype(str).str.replace('V|E','10',regex=True),errors='coerce')
for dis,rg in DR.items():
    m=False
    for lo,hi in rg: m=m|u[['diag_1','diag_2','diag_3']].apply(lambda col:col.between(lo,hi)).any(axis=1)
    u[dis]=m.astype(int)
MED=['metformin','repaglinide','nateglinide','chlorpropamide','glimepiride','acetohexamide','glipizide',
 'glyburide','tolbutamide','pioglitazone','rosiglitazone','acarbose','miglitol','troglitazone','tolazamide',
 'examide','citoglipton','insulin','glyburide-metformin','glipizide-metformin','glimepiride-pioglitazone',
 'metformin-rosiglitazone','metformin-pioglitazone']
med=[c for c in MED if c in u.columns]
u['med_change_count']=u[med].isin(['Up','Down']).sum(axis=1)
u['comorbidity_count']=u[list(DR)].sum(axis=1)
u['total_prior_visits']=(u['number_inpatient'].fillna(0)+u['number_emergency'].fillna(0)+u['number_outpatient'].fillna(0))
u=u.drop(columns=['diag_1','diag_2','diag_3'])
amap={'[0-10)':None,'[10-20)':None,'[20-30)':'20-39','[30-40)':'20-39','[40-50)':'40-59','[50-60)':'40-59',
 '[60-70)':'>=60','[70-80)':'>=60','[80-90)':'>=60','[90-100)':'>=60'}
u['age']=u['age'].map(amap); u=u.dropna(subset=['age'])
u=u[u['gender'].isin(['Male','Female'])]
u['gender']=u['gender'].map({'Male':1,'Female':2}).astype(int)
u['race']=u['race'].map({'Caucasian':3,'AfricanAmerican':4,'Hispanic':2,'Asian':5,'Other':5}).fillna(5).astype(int)
u['readmitted']=(u['readmitted'].astype(str)=='<30').astype(int)
ENC=u['encounter_id'].values.copy()          # kept only for first-encounter ordering
u=u.drop(columns=[c for c in ['encounter_id','weight','payer_code','medical_specialty'] if c in u.columns])
u['patient_nbr']=u['patient_nbr'].astype(int)
base=u.drop(columns=['on_insulin'],errors='ignore').reset_index(drop=True)
assert (len(base),base.patient_nbr.nunique(),int(base.readmitted.sum()))==(98490,69311,11271)
print('cohort matches:',len(base),base.patient_nbr.nunique(),int(base.readmitted.sum()))
AGE=base['age'].values.copy()

In [ ]:
#@title 2. Encode, model helper, baseline out-of-fold predictions
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

CAT=['admission_type_id','discharge_disposition_id','admission_source_id','race']
def encode(df):
    y=df['readmitted'].values; g=df['patient_nbr'].values
    X=df.drop(columns=['readmitted','patient_nbr'])
    for c in CAT:
        if c in X.columns: X[c]=X[c].astype(str)
    cat=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    return pd.get_dummies(X,columns=cat,dummy_na=False).astype(float).values.astype(float),y,g

def mk(pw,seed=42):
    kw=dict(n_estimators=500,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.7,
            min_child_weight=5,reg_lambda=2.0,scale_pos_weight=pw,eval_metric='logloss',
            random_state=seed,tree_method='hist')
    if USE_GPU: kw['device']='cuda'
    return XGBClassifier(**kw)

EPS=1e-9
def logit(p): p=np.clip(p,EPS,1-EPS); return np.log(p/(1-p))

def oof_predictions(Xv,y,g,seed=42,recalibrate=False):
    """Patient-grouped 5-fold, class weighting. If recalibrate, Platt-scale using a
       patient-grouped 20% slice of the training fold that never touches the test fold."""
    pw=(y==0).sum()/max((y==1).sum(),1)
    oof=np.zeros(len(y))
    for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(y)),y,g):
        if not recalibrate:
            sc=StandardScaler().fit(Xv[tr]); m=mk(pw,seed); m.fit(sc.transform(Xv[tr]),y[tr])
            oof[te]=m.predict_proba(sc.transform(Xv[te]))[:,1]
        else:
            tp=np.unique(g[tr]); rs=np.random.default_rng(seed)
            cal=set(rs.choice(tp,size=int(0.2*len(tp)),replace=False))
            isc=np.array([gg in cal for gg in g[tr]])
            fit,cali=tr[~isc],tr[isc]
            sc=StandardScaler().fit(Xv[fit]); m=mk(pw,seed); m.fit(sc.transform(Xv[fit]),y[fit])
            pc=m.predict_proba(sc.transform(Xv[cali]))[:,1]
            pt=m.predict_proba(sc.transform(Xv[te]))[:,1]
            pl=LogisticRegression().fit(logit(pc).reshape(-1,1),y[cali])
            oof[te]=pl.predict_proba(logit(pt).reshape(-1,1))[:,1]
    return oof

Xv,y,g=encode(base)
print('encoded predictors:',Xv.shape[1],'(expect 155)')
oof_raw=oof_predictions(Xv,y,g,recalibrate=False)
oof_cal=oof_predictions(Xv,y,g,recalibrate=True)
print(f'AUROC raw {roc_auc_score(y,oof_raw):.4f} | recalibrated {roc_auc_score(y,oof_cal):.4f}')
print(f'Brier raw {brier_score_loss(y,oof_raw):.4f} | recalibrated {brier_score_loss(y,oof_cal):.4f}')

## A. Decision curve analysis

In [ ]:
#@title 3. Net benefit across threshold probabilities
import matplotlib; import matplotlib.pyplot as plt

def net_benefit(y,p,pt):
    n=len(y); flag=p>=pt
    tp=np.sum(flag&(y==1)); fp=np.sum(flag&(y==0))
    return tp/n - (fp/n)*(pt/(1-pt))

prev=y.mean()
ths=np.arange(0.01,0.51,0.005)
nb_model=np.array([net_benefit(y,oof_cal,t) for t in ths])
nb_all  =np.array([prev-(1-prev)*(t/(1-t)) for t in ths])
nb_none =np.zeros_like(ths)

# range over which the model beats both defaults
better=(nb_model>nb_all)&(nb_model>nb_none)
lo,hi=(ths[better].min(),ths[better].max()) if better.any() else (np.nan,np.nan)
print(f'model has the highest net benefit for threshold probabilities from {lo:.3f} to {hi:.3f}')

rows=[]
for t in [0.10,0.15,0.20,0.25,0.30]:
    nbm=net_benefit(y,oof_cal,t); nba=prev-(1-prev)*(t/(1-t))
    # interventions avoided per 100 relative to treat-all, at equal true positives
    avoided=(nbm-nba)/(t/(1-t))*100
    rows.append(dict(threshold=t, net_benefit_model=round(nbm,5), net_benefit_treat_all=round(nba,5),
                     net_benefit_treat_none=0.0,
                     interventions_avoided_per_100=round(avoided,1)))
dca=pd.DataFrame(rows); display(dca)

plt.rcParams.update({'font.family':'serif','font.size':10})
fig,ax=plt.subplots(figsize=(6.4,4.0))
ax.plot(ths,nb_model,color='#1f3864',lw=2,label='Baseline XGBoost (recalibrated)')
ax.plot(ths,nb_all,color='#b3261e',ls='--',lw=1.2,label='Flag all discharges')
ax.plot(ths,nb_none,color='0.4',ls=':',lw=1.2,label='Flag none')
ax.set_xlabel('Threshold probability'); ax.set_ylabel('Net benefit')
ax.set_ylim(-0.02,max(nb_model.max(),prev)*1.15); ax.set_xlim(ths.min(),ths.max())
ax.legend(frameon=False,fontsize=8); ax.spines[['top','right']].set_visible(False)
ax.set_title('Decision curve, whole cohort',fontsize=11)
plt.tight_layout(); plt.savefig('fig9_decision_curve.png',dpi=300); plt.show()

## B. First-encounter-only sensitivity analysis

In [ ]:
#@title 4. Restrict to each patient's first encounter
#@markdown encounter_id increases with admission order in this dataset, so the smallest
#@markdown encounter_id per patient identifies the first recorded encounter.
order=np.argsort(ENC)
first_mask=np.zeros(len(base),dtype=bool)
seen=set()
for i in order:
    p=base.patient_nbr.values[i]
    if p not in seen: first_mask[i]=True; seen.add(p)
fe=base[first_mask].reset_index(drop=True)
print(f'first-encounter analysis set: {len(fe)} encounters, {fe.patient_nbr.nunique()} patients, '
      f'{int(fe.readmitted.sum())} events, prevalence {fe.readmitted.mean():.4f}')

Xf,yf,gf=encode(fe)
oof_fe=oof_predictions(Xf,yf,gf,recalibrate=False)
auc_fe=roc_auc_score(yf,oof_fe)
print(f'\nfirst-encounter AUROC {auc_fe:.4f}')
print(f'full-cohort   AUROC {roc_auc_score(y,oof_raw):.4f}')
print(f'difference {auc_fe-roc_auc_score(y,oof_raw):+.4f}')
print('\nWith one encounter per patient there is no repeated-measures structure at all,')
print('so patient-grouped and encounter-level splitting are identical by construction.')

## C. SMOTE neighbour sensitivity

In [ ]:
#@title 5. Does the inflation depend on k?
def presplit_smote_auc(Xv,y,g,k,seed=42):
    Xr,yr=SMOTE(random_state=seed,k_neighbors=k).fit_resample(Xv,y)
    gr=np.concatenate([g,np.arange(g.max()+1,g.max()+1+(len(yr)-len(y)))])
    oof=np.zeros(len(yr))
    for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=seed).split(np.zeros(len(yr)),yr,gr):
        sc=StandardScaler().fit(Xr[tr]); m=mk(1.0,seed); m.fit(sc.transform(Xr[tr]),yr[tr])
        oof[te]=m.predict_proba(sc.transform(Xr[te]))[:,1]
    return roc_auc_score(yr,oof)

base_auc=roc_auc_score(y,oof_raw)
ks=[1,5,15]  #@param
rows=[]
for k in ks:
    a=presplit_smote_auc(Xv,y,g,k)
    rows.append(dict(k_neighbors=k,auroc=round(a,4),inflation=round(a-base_auc,4)))
    print(f'  k={k:2d}  AUROC={a:.4f}  inflation {a-base_auc:+.4f}',flush=True)
smote_k=pd.DataFrame(rows); display(smote_k)

In [ ]:
#@title 6. Summary and paste-ready sentences
lines=[]
def log(s): print(s); lines.append(s)
log(f'environment: xgboost {xgb.__version__}, sklearn {sklearn.__version__}, GPU {USE_GPU}')
log('')
log('A. DECISION CURVE')
log(dca.to_string(index=False))
log(f'  model has the highest net benefit from threshold {lo:.3f} to {hi:.3f}')
log('')
log('B. FIRST-ENCOUNTER SENSITIVITY')
log(f'  {len(fe)} encounters, {int(fe.readmitted.sum())} events, prevalence {fe.readmitted.mean():.4f}')
log(f'  AUROC {auc_fe:.4f} vs {base_auc:.4f} on the full cohort ({auc_fe-base_auc:+.4f})')
log('')
log('C. SMOTE NEIGHBOUR SENSITIVITY')
log(smote_k.to_string(index=False))
open('optional_analyses.txt','w').write('\n'.join(lines))
dca.to_csv('decision_curve.csv',index=False); smote_k.to_csv('smote_k.csv',index=False)
from google.colab import files
files.download('optional_analyses.txt'); files.download('fig9_decision_curve.png')